In [6]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Main repository directory (one level above src)
BASE_DIR = Path.cwd().parent

# Data and output folders in the main repository
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
MODEL_DIR = BASE_DIR / "models"

# Dataset paths
DATA_PATH = DATA_DIR / "diabetic_data.csv"
MAPPING_PATH = DATA_DIR / "IDS_mapping.csv"



### Loading the preprocessed data from the output folder

In [7]:
X_train = pd.read_csv(OUTPUT_DIR / "X_train.csv")
X_test = pd.read_csv(OUTPUT_DIR / "X_test.csv")

y_train = pd.read_csv(OUTPUT_DIR / "y_train.csv")["target"]
y_test = pd.read_csv(OUTPUT_DIR / "y_test.csv")["target"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

display(X_train.head())

Training features: (81613, 45)
Testing features: (20153, 45)

Training target distribution:
target
0    72407
1     9206
Name: count, dtype: int64

Testing target distribution:
target
0    18002
1     2151
Name: count, dtype: int64


,race,gender,age,weight,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,admission_type,discharge_disposition,admission_source
0,Caucasian,Female,[0-10),NaN,1,NaN,Pediatrics-Endocrinology,41,0,1,...,No,No,No,No,No,No,No,NaN,Not Mapped,Physician Referral
1,Caucasian,Female,[10-20),NaN,3,NaN,NaN,59,0,18,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room
2,AfricanAmerican,Female,[20-30),NaN,2,NaN,NaN,11,5,13,...,No,No,No,No,No,No,Yes,Emergency,Discharged to home,Emergency Room
3,Caucasian,Male,[30-40),NaN,2,NaN,NaN,44,1,16,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room
4,Caucasian,Male,[40-50),NaN,1,NaN,NaN,51,0,8,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room


### Identifying numerical and categorical features

In [8]:

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 8
Number of categorical features: 37

Numerical features:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features:
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type', 'discharge_disposition', 'admission_source']
